In [5]:
from pathlib import Path
import pandas as pd

# Locate the project root whether Jupyter starts in the project root or
# inside the notebook directory. This repository stores files under data/.
NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SEARCH_DIRS = [Path.cwd(), *Path.cwd().parents, NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
ROOT = next((d for d in SEARCH_DIRS if (d / "data" / "raw").is_dir() or (d / "datasets" / "raw").is_dir()), Path.cwd())
DATA_DIR = ROOT / ("data" if (ROOT / "data" / "raw").is_dir() else "datasets")
RAW = DATA_DIR / "raw"
OUT = DATA_DIR / "processed"

pair_file = (
    RAW / "AgriMind_Bilingual_Paddy_Farmer_QA_12720_Pairs.csv"
)

rows_file = (
    RAW / "AgriMind_Sinhala_English_Paddy_Farmer_QA_25440.csv"
)

pairs = pd.read_csv(
    pair_file,
    keep_default_na=False,
    encoding="utf-8"
)

rows = pd.read_csv(
    rows_file,
    keep_default_na=False,
    encoding="utf-8"
)

metadata = [
    "pair_id",
    "category_en",
    "category_si",
    "issue_id",
    "issue_en",
    "crop_stage_en",
    "crop_stage_si",
    "urgency",
    "source_url",
    "data_origin",
    "review_status",
]

# Create Sinhala records.
si = pairs[
    metadata + ["question_si", "answer_si"]
].copy()

si = si.rename(
    columns={
        "question_si": "question",
        "answer_si": "answer",
    }
)

si["language"] = "si-LK"
si["qa_id"] = si["pair_id"].str.replace(
    "PADDY_PAIR_",
    "PADDY_SI_",
    regex=False,
)
si["is_real_farmer_query"] = False

# Create English records.
en = pairs[
    metadata + ["question_en", "answer_en"]
].copy()

en = en.rename(
    columns={
        "question_en": "question",
        "answer_en": "answer",
    }
)

en["language"] = "en-LK"
en["qa_id"] = en["pair_id"].str.replace(
    "PADDY_PAIR_",
    "PADDY_EN_",
    regex=False,
)
en["is_real_farmer_query"] = False

combined_from_pairs = pd.concat(
    [si, en],
    ignore_index=True,
)

# Check both CSV representations match.
comparison_columns = [
    "question",
    "answer",
    "category_en",
    "category_si",
    "issue_id",
    "issue_en",
    "crop_stage_en",
    "crop_stage_si",
    "urgency",
    "source_url",
    "data_origin",
    "review_status",
]

left = combined_from_pairs[
    ["pair_id", "language"] + comparison_columns
].copy()

right = rows[
    ["pair_id", "language"] + comparison_columns
].copy()

for frame in (left, right):
    for col in frame.select_dtypes(include="object"):
        frame[col] = frame[col].astype(str).str.strip()

check = left.merge(
    right,
    on=["pair_id", "language"],
    how="outer",
    suffixes=("_pair", "_rows"),
    indicator=True,
    validate="one_to_one",
)

assert (check["_merge"] == "both").all()

for col in comparison_columns:
    assert (
        check[f"{col}_pair"] == check[f"{col}_rows"]
    ).all(), f"Mismatch in {col}"

# Use the language-specific CSV as the canonical output.
combined = rows.copy()

for col in combined.select_dtypes(include="object"):
    combined[col] = combined[col].astype(str).str.strip()

combined["is_real_farmer_query"] = (
    combined["is_real_farmer_query"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

combined = combined[
    [
        "qa_id",
        "pair_id",
        "question",
        "answer",
        "language",
        "category_en",
        "category_si",
        "issue_id",
        "issue_en",
        "crop_stage_en",
        "crop_stage_si",
        "urgency",
        "source_url",
        "data_origin",
        "review_status",
        "is_real_farmer_query",
    ]
]

assert not combined.duplicated(
    ["pair_id", "language"]
).any()

assert combined["pair_id"].nunique() == 12720

assert combined["language"].value_counts().to_dict() == {
    "si-LK": 12720,
    "en-LK": 12720,
}

OUT.mkdir(parents=True, exist_ok=True)

combined.to_csv(
    OUT / "multilingual_paddy_qa.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Combined dataset created.")
print("Shape:", combined.shape)
print(combined["language"].value_counts())
print("Unique bilingual pairs:", combined["pair_id"].nunique())

C:\Users\USER\AppData\Local\Temp\ipykernel_26940\2124983979.py:117: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in frame.select_dtypes(include="object"):
C:\Users\USER\AppData\Local\Temp\ipykernel_26940\2124983979.py:117: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migra

Combined dataset created.
Shape: (25440, 16)
language
si-LK    12720
en-LK    12720
Name: count, dtype: int64
Unique bilingual pairs: 12720
